# 3 — RAG corpus ingestion (Vertex AI RAG Engine)

**AWS-side equivalent:** Titan Embeddings V2 + pgvector storage in Aurora
**GCP service:** Vertex AI RAG Engine — corpus + ingestion as a managed
service. Embeddings are computed internally by the corpus (we do **not**
call `embed_content` ourselves the way the AWS notebook does).

AWS-side notebook: `notebooks/aws/03-embeddings.ipynb`

> **API note (verified May 2026):** `vertexai.rag` is GA and is the path forward.
> The companion `vertexai.generative_models` namespace deprecates **2026-06-24**;
> for now we use it in `04b-rag-engine.ipynb` for the `Tool.from_retrieval`
> wiring, and will migrate to the unified `google-genai` SDK once it ships
> the equivalent surface.
>
> **Embedding model constraint:** RAG Engine only supports `text-embedding-005`
> or `text-embedding-004`. The corpus-managed embedding stack does **not**
> accept `gemini-embedding-001` (despite our raw `embed_content` notebooks
> using that model). We pin to `text-embedding-005` here.


## Step 0: Imports + config

In [25]:
import json
from pathlib import Path

import vertexai
from vertexai import rag
from google.cloud import bigquery, storage
from google.api_core.exceptions import Conflict, NotFound

from cleanup_utils import ResourceTracker

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})

PROJECT_ID    = gcp.get('project_id')
REGION        = gcp.get('region', 'us-central1')
BQ_DATASET    = gcp.get('bq_dataset', 'orgiq')
STAGING_BUCKET = gcp.get('staging_bucket', f'gs://{PROJECT_ID}-orgiq-staging')
CORPUS_NAME   = gcp.get('rag_corpus_display_name', 'orgiq-people-corpus')
CHUNK_SIZE    = gcp.get('rag_chunk_size', 512)
CHUNK_OVERLAP = gcp.get('rag_chunk_overlap', 100)

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

vertexai.init(project=PROJECT_ID, location=REGION)
tracker = ResourceTracker()

print(f'project        : {PROJECT_ID}')
print(f'region         : {REGION}')
print(f'BQ dataset     : {BQ_DATASET}')
print(f'staging bucket : {STAGING_BUCKET}')
print(f'corpus display : {CORPUS_NAME}')
print(f'chunk size     : {CHUNK_SIZE} (overlap {CHUNK_OVERLAP})')

project        : hoehncloudconsole
region         : us-central1
BQ dataset     : orgiq
staging bucket : gs://hoehncloudconsole-orgiq-staging
corpus display : orgiq-people-corpus
chunk size     : 512 (overlap 100)


## Step 1: Read people from BigQuery

Mirrors the AWS-side query against `people` in Aurora. If the BigQuery
table is missing (e.g. notebook 02 hasn't run), fall back to the local
seed file so this notebook still works in isolation.

In [26]:
def load_people() -> list[dict]:
    """Try BigQuery first, fall back to the local seed file."""
    try:
        bq = bigquery.Client(project=PROJECT_ID)
        query = f"""
            SELECT id, name, title, department, manager_id
            FROM `{PROJECT_ID}.{BQ_DATASET}.people`
        """
        rows = [dict(r) for r in bq.query(query).result()]
        if rows:
            print(f'✓ Loaded {len(rows)} people from BigQuery')
            return rows
        print('  BigQuery returned 0 rows; falling back to seed file')
    except Exception as exc:  # noqa: BLE001
        print(f'  BigQuery read failed ({type(exc).__name__}); using seed file')

    seed = json.loads(Path('data/org/people.json').read_text())
    print(f'✓ Loaded {len(seed)} people from data/org/people.json')
    return seed

people = load_people()
people[0]

✓ Loaded 20 people from BigQuery


{'id': 'p-005',
 'name': 'David Kim',
 'title': 'Engineering Manager',
 'department': 'Platform Engineering',
 'manager_id': None}

## Step 2: Compose person-as-document markdown

One markdown file per person — RAG Engine handles chunking + embedding
internally during `import_files`. The corpus will end up with `N` source
files and (post-chunking) some multiple of that as retrievable nodes.

We also load `teams.json` so we can put each person on their team and
mention the team's charter — that gives the retriever some semantic
ballast beyond just job titles.

In [27]:
# Cross-reference teams so we can write "{name} is on the {team} team" lines.
teams_raw = json.loads(Path('data/org/teams.json').read_text())
team_by_member = {}
for t in teams_raw:
    for member_id in t.get('member_ids', []):
        team_by_member[member_id] = t

def person_to_markdown(p: dict) -> str:
    team = team_by_member.get(p['id'])
    lines = [
        f"# {p['name']}",
        '',
        f"**Title:** {p.get('title', 'unknown')}",
        f"**Department:** {p.get('department', 'unknown')}",
    ]
    if p.get('manager_id'):
        lines.append(f"**Manager:** {p['manager_id']}")
    if team:
        lines += [
            '',
            f"## Team — {team['name']}",
            '',
            team.get('charter', ''),
        ]
    lines += [
        '',
        '## Role summary',
        '',
        f"{p['name']} is a {p.get('title', 'team member')} in {p.get('department', 'the org')}. "
        + (f"Works on the {team['name']} team. " if team else '')
        + 'Reach out for questions about their domain.',
    ]
    return '\n'.join(lines).strip() + '\n'

# Write the JSONL ledger + per-person markdown to /tmp.
jsonl_path = Path('/tmp/orgiq-people.jsonl')
md_dir = Path('/tmp/orgiq-people-md')
md_dir.mkdir(exist_ok=True)

with jsonl_path.open('w') as f:
    for p in people:
        body = person_to_markdown(p)
        (md_dir / f"{p['id']}.md").write_text(body)
        f.write(json.dumps({'id': p['id'], 'name': p['name'], 'body': body}) + '\n')

print(f'✓ Wrote {len(people)} markdown files to {md_dir}/')
print(f'✓ Wrote JSONL ledger to {jsonl_path}')
print('\n--- sample ---')
print((md_dir / f"{people[0]['id']}.md").read_text())

✓ Wrote 20 markdown files to /tmp/orgiq-people-md/
✓ Wrote JSONL ledger to /tmp/orgiq-people.jsonl

--- sample ---
# David Kim

**Title:** Engineering Manager
**Department:** Platform Engineering

## Team — Platform Engineering

Build and maintain core platform infrastructure including APIs, authentication, CI/CD pipelines, and developer tooling.

## Role summary

David Kim is a Engineering Manager in Platform Engineering. Works on the Platform Engineering team. Reach out for questions about their domain.



## Step 3: Upload to the GCS staging bucket

The corpus reads files from GCS. We use a stable prefix `orgiq-corpus/people/`
inside the staging bucket so 19-cleanup can purge it cheaply (deleting the
corpus drops the embeddings but leaves the source files in place).

In [28]:
# STAGING_BUCKET may be a bare name or a gs://… URI — normalize.
bucket_name = STAGING_BUCKET[5:].split('/', 1)[0] if STAGING_BUCKET.startswith('gs://') else STAGING_BUCKET

gcs = storage.Client(project=PROJECT_ID)
try:
    bucket = gcs.create_bucket(bucket_name, location='US')
    print(f'✓ Created bucket gs://{bucket_name}')
except Conflict:
    bucket = gcs.bucket(bucket_name)
    print(f'  Bucket gs://{bucket_name} already exists')

tracker.track('gcs_bucket', bucket_name)
tracker.flush()

prefix = 'orgiq-corpus/people'
for md_file in sorted(md_dir.glob('*.md')):
    blob = bucket.blob(f'{prefix}/{md_file.name}')
    blob.upload_from_string(md_file.read_text(), content_type='text/markdown')

print(f'✓ Uploaded {len(list(md_dir.glob("*.md")))} files to gs://{bucket_name}/{prefix}/')
CORPUS_GCS_URI = f'gs://{bucket_name}/{prefix}/'
print(f'  corpus source URI: {CORPUS_GCS_URI}')

✓ Created bucket gs://hoehncloudconsole-orgiq-staging
✓ Uploaded 20 files to gs://hoehncloudconsole-orgiq-staging/orgiq-corpus/people/
  corpus source URI: gs://hoehncloudconsole-orgiq-staging/orgiq-corpus/people/


## Step 4: Create (or reuse) the RAG corpus

`rag.create_corpus` does **not** dedupe on `display_name` — calling it
twice creates two corpora. We list first, reuse on hit, create on miss.

**Why the two-step approach:** New projects default to Spanner mode, which is
allowlist-only for `us-central1 / us-east1 / us-east4`. Serverless mode is
available in `us-central1` but must be explicitly enabled **at the project level**
first via `UpdateRagEngineConfig`, then corpus creation is a plain `displayName`-only
call. Both calls use `v1beta1` (the serverless fields are not yet in `v1`).

The `UpdateRagEngineConfig` PATCH is idempotent — safe to re-run.


In [29]:
import time
import requests
import google.auth
import google.auth.transport.requests


def _auth_header() -> dict:
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    return {'Authorization': f'Bearer {creds.token}'}


def _poll_lro(op_name: str, label: str = 'LRO') -> dict:
    op_url = f'https://{REGION}-aiplatform.googleapis.com/v1beta1/{op_name}'
    for tick in range(60):
        op = requests.get(op_url, headers=_auth_header()).json()
        if op.get('done'):
            return op
        if tick % 6 == 0:
            print(f'  {label}: {tick * 5}s elapsed…')
        time.sleep(5)
    raise TimeoutError(f'{label} timed out after 5 min')


def enable_serverless_mode():
    url = (
        f'https://{REGION}-aiplatform.googleapis.com/v1beta1'
        f'/projects/{PROJECT_ID}/locations/{REGION}/ragEngineConfig'
    )
    resp = requests.patch(
        url, json={'ragManagedDbConfig': {'serverless': {}}}, headers=_auth_header()
    )
    if not resp.ok:
        print(f'UpdateRagEngineConfig {resp.status_code}: {resp.text}')
    resp.raise_for_status()
    data = resp.json()
    if '/operations/' in data.get('name', ''):
        op = _poll_lro(data['name'], 'mode-switch')
        if 'error' in op:
            raise RuntimeError(f'mode-switch failed: {op["error"]}')
    print('✓ Serverless mode active')


def reset_corpus(display_name: str):
    """Delete any existing corpus with this name, then create a fresh one."""
    for c in rag.list_corpora():
        if c.display_name == display_name:
            print(f'  Deleting existing corpus {c.name} …')
            rag.delete_corpus(c.name)
            print('  Deleted.')
            time.sleep(5)

    url = (
        f'https://{REGION}-aiplatform.googleapis.com/v1beta1'
        f'/projects/{PROJECT_ID}/locations/{REGION}/ragCorpora'
    )
    for attempt in range(1, 4):
        resp = requests.post(url, json={'displayName': display_name}, headers=_auth_header())
        if not resp.ok:
            print(f'CreateRagCorpus {resp.status_code}: {resp.text}')
        resp.raise_for_status()

        op = _poll_lro(resp.json()['name'], f'create-corpus attempt {attempt}')
        err = op.get('error', {})
        if err:
            if err.get('code') == 13:
                print(f'  INTERNAL on attempt {attempt}/3, retrying in 20 s…')
                time.sleep(20)
                continue
            raise RuntimeError(f'corpus creation failed: {err}')
        break
    else:
        raise RuntimeError('Corpus creation failed after 3 attempts')

    # LRO name uses project number; list_corpora uses project ID — compare by corpus ID only.
    corpus_id = op['response']['name'].split('/')[-1]
    print(f'✓ Corpus created: {corpus_id}')
    for c in rag.list_corpora():
        if c.name.split('/')[-1] == corpus_id:
            return c
    raise RuntimeError(f'Corpus {corpus_id} created but not found in list')


enable_serverless_mode()
corpus = reset_corpus(CORPUS_NAME)

tracker.track('rag_corpus', corpus.name, project=PROJECT_ID, location=REGION)
tracker.flush()

gcp['rag_corpus_name'] = corpus.name
config_path.write_text(json.dumps(config, indent=2))
print(f'  corpus name: {corpus.name}')


✓ Serverless mode active
  Deleting existing corpus projects/hoehncloudconsole/locations/us-central1/ragCorpora/3857297896470740992 …
Successfully deleted the RagCorpus.
  Deleted.
  Deleting existing corpus projects/hoehncloudconsole/locations/us-central1/ragCorpora/5792719836333211648 …
Successfully deleted the RagCorpus.
  Deleted.
  create-corpus attempt 1: 0s elapsed…
✓ Corpus created: 6264524673858404352
  corpus name: projects/hoehncloudconsole/locations/us-central1/ragCorpora/6264524673858404352


## Step 5: Import the staged files into the corpus

`rag.import_files` returns synchronously once the LRO finishes (under the
hood it polls for us). For the workshop's 20-person corpus this is
typically under 2 minutes. The `max_embedding_requests_per_min` knob
keeps us safely under the embedding-model quota.

In [30]:
print(f'Importing from : {CORPUS_GCS_URI}')
print(f'Into corpus    : {corpus.name}')

import_url = (
    f'https://{REGION}-aiplatform.googleapis.com/v1beta1'
    f'/{corpus.name}/ragFiles:import'
)
# Omit ragFileTransformationConfig — the chunking config was accepted by the
# API but caused a backend INTERNAL after 4+ minutes. Use service defaults.
import_body = {
    'importRagFilesConfig': {
        'gcsSource': {'uris': [CORPUS_GCS_URI]},
        'maxEmbeddingRequestsPerMin': 1000,
    },
}

resp = requests.post(import_url, json=import_body, headers=_auth_header())
if not resp.ok:
    print(f'ImportRagFiles error {resp.status_code}:')
    print(resp.text)
resp.raise_for_status()

op_name = resp.json()['name']
print(f'  LRO started: {op_name}')

op_url = f'https://{REGION}-aiplatform.googleapis.com/v1beta1/{op_name}'
for tick in range(120):
    op = requests.get(op_url, headers=_auth_header()).json()
    if op.get('done'):
        break
    meta = op.get('metadata', {}).get('genericMetadata', {})
    print(f'  [{tick*5:4d}s] running… updateTime={meta.get("updateTime", "—")}')
    time.sleep(5)
else:
    raise TimeoutError('Import timed out after 10 min')

if 'error' in op:
    print(f'✗ Import LRO failed: {op["error"]}')
    raise RuntimeError(op['error'])

print('✓ Import complete')
print(f'  response : {op.get("response")}')
print(f'  metadata : {op.get("metadata")}')


Importing from : gs://hoehncloudconsole-orgiq-staging/orgiq-corpus/people/
Into corpus    : projects/hoehncloudconsole/locations/us-central1/ragCorpora/6264524673858404352
  LRO started: projects/289864904502/locations/us-central1/ragCorpora/6264524673858404352/operations/6125698987115675648
  [   0s] running… updateTime=2026-05-21T13:08:51.223409Z
  [   5s] running… updateTime=2026-05-21T13:08:53.210544Z
  [  10s] running… updateTime=2026-05-21T13:09:01.958276Z
  [  15s] running… updateTime=2026-05-21T13:09:01.958276Z
  [  20s] running… updateTime=2026-05-21T13:09:01.958276Z
  [  25s] running… updateTime=2026-05-21T13:09:01.958276Z
  [  30s] running… updateTime=2026-05-21T13:09:01.958276Z
✓ Import complete
  response : {'@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.ImportRagFilesResponse', 'importedRagFilesCount': '20'}
  metadata : {'@type': 'type.googleapis.com/google.cloud.aiplatform.v1beta1.ImportRagFilesOperationMetadata', 'genericMetadata': {'createTime': '2026-0

## Step 6: Verify — list files in the corpus

In [31]:
files = list(rag.list_files(corpus.name))
print(f'Corpus contains {len(files)} files:')
for f in files[:5]:
    # `display_name` typically matches the source filename.
    print(f'  - {f.display_name}  ({f.name.split("/")[-1]})')
if len(files) > 5:
    print(f'  … and {len(files) - 5} more')

Corpus contains 20 files:
  - p-002.md  (5703056438340361475)
  - p-001.md  (5703056439799633602)
  - p-003.md  (5703056439619217931)
  - p-009.md  (5703056458341016455)
  - p-008.md  (5703056460046627304)
  … and 15 more


## Summary

| Resource | Details |
|---|---|
| **Service** | Vertex AI RAG Engine (managed retrieval) |
| **Corpus** | `rag.create_corpus(display_name='orgiq-people-corpus')` |
| **Backend** | Default `RagManagedDb` — no Vector Search provisioning |
| **Embeddings** | `text-embedding-005` (managed by the corpus) |
| **Chunking** | size=512 tokens, overlap=100 |
| **Source** | `gs://<staging>/orgiq-corpus/people/*.md` (one file per person) |
| **Idle cost** | Storage + per-query at retrieval time; no provisioning fee |
| **Tracked** | `rag_corpus` (Python deleter), `gcs_bucket` (gcloud CLI deleter) |

**Next:** `04-vertex-search.ipynb` covers turnkey Discovery Engine retrieval, and
`04b-rag-engine.ipynb` uses *this* corpus for managed retrieval + Gemini grounded answers.
